# Can we predict 12-month loan default?
**Target:** `defaulted` (1 = defaulted within 12 months after funding).
**Plan:** inspect data → check ID column for signal → compare models with cross-validation → evaluate once on a held-out test set → interpret.

Metrics: the classes are imbalanced, so accuracy is misleading. We use ROC AUC and PR AUC (average precision), plus precision/recall when flagging the riskiest applicants.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, confusion_matrix, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
SEED=42
df=pd.read_csv('data.csv'); dd=pd.read_csv('columns.csv')
display(dd); df.head()

## 1. Data quality

In [ ]:
print(df.shape); print('missing:', int(df.isna().sum().sum()), '| duplicate rows:', df.duplicated().sum(), '| duplicate ids:', df.application_id.duplicated().sum())
display(df.describe().T)
rate=df.defaulted.mean(); print(f'Default rate: {rate:.2%} ({df.defaulted.sum()} of {len(df)})')
print((df[['income_k','loan_amount_k','employment_years','credit_lines']]<0).sum().to_dict(), '| age range', df.age.min(), df.age.max(),
      '| employment_years > age-16:', (df.employment_years>df.age-16).sum())

## 2. Univariate relationship with default

In [ ]:
num=['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']
fig,ax=plt.subplots(2,3,figsize=(14,7))
for a,c in zip(ax.ravel(),num):
    q=pd.qcut(df[c],10,duplicates='drop')
    g=df.groupby(q,observed=True).defaulted.mean()
    a.plot(range(len(g)),g.values,marker='o'); a.axhline(rate,ls='--',c='gray'); a.set_title(c+' (decile → default rate)'); a.set_xlabel('decile')
plt.tight_layout(); plt.show()
print(df.groupby('region').defaulted.agg(['mean','size']))

`application_id` is an identifier and should carry no signal. We verify that, then exclude it.

In [ ]:
print('AUC of application_id alone:', round(roc_auc_score(df.defaulted, df.application_id),3))

## 3. Split
There is no date column, so a time-based split isn't possible; we use a stratified random split. 20% is held out and touched only once at the end. Model selection uses 5-fold CV on the training part.

In [ ]:
cat=['region']
X=df[cat+num]; y=df.defaulted
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)
print(Xtr.shape,Xte.shape,ytr.mean().round(4),yte.mean().round(4))
pre_s=ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(handle_unknown='ignore'),cat)])
pre_t=ColumnTransformer([('n','passthrough',num),('c',OneHotEncoder(handle_unknown='ignore'),cat)])
models={
 'baseline (prior)':DummyClassifier(strategy='prior'),
 'logistic':Pipeline([('p',pre_s),('m',LogisticRegression(max_iter=1000))]),
 'random forest':Pipeline([('p',pre_t),('m',RandomForestClassifier(500,min_samples_leaf=20,n_jobs=-1,random_state=SEED))]),
 'grad. boosting':Pipeline([('p',pre_t),('m',HistGradientBoostingClassifier(learning_rate=0.05,max_iter=200,max_leaf_nodes=8,min_samples_leaf=40,l2_regularization=1.0,random_state=SEED))]),
}

## 4. Model comparison (5-fold stratified CV on training data)

In [ ]:
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
rows=[]
for n,m in models.items():
    r=cross_validate(m,Xtr,ytr,cv=cv,scoring={'auc':'roc_auc','ap':'average_precision'})
    rows.append({'model':n,'AUC':r['test_auc'].mean(),'AUC sd':r['test_auc'].std(),'PR-AUC':r['test_ap'].mean(),'PR-AUC sd':r['test_ap'].std()})
cvres=pd.DataFrame(rows).set_index('model').round(3); cvres

## 5. Final evaluation on the held-out test set

In [ ]:
use_lr = cvres.loc['logistic','AUC'] >= cvres.loc['grad. boosting','AUC']-0.005
best = models['logistic'] if use_lr else models['grad. boosting']
print('Chosen:', 'logistic' if use_lr else 'grad. boosting', '(simpler model preferred unless boosting is clearly better)')
best.fit(Xtr,ytr); p=best.predict_proba(Xte)[:,1]; yt=yte.values
rng=np.random.default_rng(SEED); a=[];b=[]
for _ in range(1000):
    i=rng.integers(0,len(yt),len(yt)); a.append(roc_auc_score(yt[i],p[i])); b.append(average_precision_score(yt[i],p[i]))
print(f'Test AUC    {roc_auc_score(yt,p):.3f}  95% CI [{np.percentile(a,2.5):.3f}, {np.percentile(a,97.5):.3f}]')
print(f'Test PR-AUC {average_precision_score(yt,p):.3f}  95% CI [{np.percentile(b,2.5):.3f}, {np.percentile(b,97.5):.3f}]  (no-skill = {yt.mean():.3f})')
print(f'Brier {brier_score_loss(yt,p):.4f} vs. prior-only {brier_score_loss(yt,np.full(len(yt),ytr.mean())):.4f}')

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4.5))
f,t,_=roc_curve(yt,p); ax[0].plot(f,t); ax[0].plot([0,1],[0,1],'--',c='gray'); ax[0].set_title('ROC'); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR')
pr,rc,_=precision_recall_curve(yt,p); ax[1].plot(rc,pr); ax[1].axhline(yt.mean(),ls='--',c='gray'); ax[1].set_title('Precision-Recall'); ax[1].set_xlabel('recall'); ax[1].set_ylabel('precision')
fp,mp=calibration_curve(yt,p,n_bins=8,strategy='quantile'); ax[2].plot(mp,fp,marker='o'); m=max(mp.max(),fp.max())*1.1; ax[2].plot([0,m],[0,m],'--',c='gray'); ax[2].set_title('Calibration'); ax[2].set_xlabel('predicted'); ax[2].set_ylabel('observed')
plt.tight_layout(); plt.show()

### Operational view: rank applicants by predicted risk

In [ ]:
t=pd.DataFrame({'p':p,'y':yt}); t['band']=pd.qcut(t.p.rank(method='first'),5,labels=['1 lowest risk','2','3','4','5 highest risk'])
lift=t.groupby('band',observed=True).agg(n=('y','size'),default_rate=('y','mean'),mean_pred=('p','mean')).round(3); display(lift)
print('Lift of top quintile vs. base rate: %.2fx'%(lift.default_rate.iloc[-1]/yt.mean()))
rows=[]
for k in [0.05,0.1,0.2,0.3]:
    th=np.quantile(p,1-k); pred=p>=th; tn,fp_,fn,tp=confusion_matrix(yt,pred).ravel()
    rows.append({'flag top':f'{k:.0%}','precision':tp/(tp+fp_),'recall':tp/(tp+fn)})
pd.DataFrame(rows).round(3)

## 6. Which features drive the prediction?

In [ ]:
pi=permutation_importance(best,Xte,yte,scoring='roc_auc',n_repeats=20,random_state=SEED)
display(pd.DataFrame({'AUC drop':pi.importances_mean,'sd':pi.importances_std},index=Xte.columns).sort_values('AUC drop',ascending=False).round(4))
if use_lr:
    names=best.named_steps['p'].get_feature_names_out(); co=pd.Series(best.named_steps['m'].coef_[0],index=names).sort_values()
    display(co.round(3).rename('std. coefficient (log-odds)').to_frame())

## 7. Conclusions (numbers from a run with seed 42; re-running should reproduce them)
- **Yes, default is predictable to a useful but modest degree.** Held-out ROC AUC ≈ 0.78 (95% CI ≈ 0.74–0.81), PR-AUC ≈ 0.35 vs. 0.10 for a no-skill model. The default rate is ~10.5%.
- **Simple beats complex here.** Logistic regression (CV AUC 0.776) matched or beat random forest (0.766) and gradient boosting (0.758), which suggests the signal is mostly additive/monotone.
- **Ranking works well; individual calls are uncertain.** The riskiest 20% of applicants default at ~28% (≈2.7x base rate), the safest 20% at ~1%. Flagging the top 10% gives ~36% precision and ~35% recall. Predicted probabilities are reasonably calibrated.
- **Drivers:** `debt_ratio` (higher → riskier), `income_k` and `employment_years` (higher → safer), `loan_amount_k` (larger → riskier). `region`, `age` and `credit_lines` add essentially nothing.
- **Caveats:**
  - No date column, so the evaluation is a random split, not out-of-time; performance on future cohorts may be lower if the population drifts.
  - Only applications with a funded loan are present, so this says nothing about rejected applicants (selection bias).
  - 643 rows have `employment_years` > age − 16, which looks like a data-quality issue worth checking with the data owner.
  - The cut-off (what share to flag) should come from the cost of a missed default vs. a wrongly declined loan, which we don't have.
  - `application_id` was excluded (AUC alone ≈ 0.50, i.e. no signal).